<a href="https://colab.research.google.com/github/muhammad-asif10/gsoc-2026-renaIssance/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive

This cell mounts the user's Google Drive to the Colab environment. This step is crucial for accessing datasets, saving models, and other project-related files stored in Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install Dependencies

This cell installs the necessary Python libraries required for the project. These include:

*   `torch`: The primary deep learning framework.
*   `torchvision`: A package supporting computer vision in PyTorch.
*   `opencv-python`: OpenCV library for image processing tasks.
*   `tqdm`: A fast, extensible progress bar for Python.

The `-q` flag ensures a quiet installation, suppressing verbose output.

In [ ]:
!pip install torch torchvision opencv-python tqdm -q

### Import Libraries and Define Core Components

This cell performs several critical setup steps:

1.  **Mount Google Drive**: Re-mounts Google Drive, ensuring continued access to external files.
2.  **Import Core Libraries**: Imports all necessary modules for the OCR model, including `torch`, `torch.nn`, `torch.optim`, `Dataset`, `DataLoader`, `cv2`, `numpy`, `logging`, `pathlib`, `tqdm`, `time`, and `json`.
3.  **Setup Logging**: Configures a basic logger to provide informative output during script execution, aiding in debugging and monitoring.
4.  **`LineOCRDataset` Class**: Defines a custom `Dataset` class for loading and preprocessing OCR image-text pairs. Key features include:
    *   Loading images and corresponding text labels from specified split directories (`train`, `val`, `test`).
    *   Resizing images to a target `(width, height)`.
    *   Converting images to RGB and normalizing pixel values.
    *   Encoding text labels to numerical sequences, handling padding/truncation, and filtering characters based on `vocab_size`.
    *   Error handling for missing files or invalid data.
5.  **`CNNRNNOCRModel` Class**: Implements the OCR neural network architecture, which combines:
    *   **CNN Backbone**: Extracts features from input images using convolutional layers, batch normalization, ReLU activations, and max-pooling.
    *   **LSTM Layer**: Processes the sequential features extracted by the CNN, enabling character-level prediction. It uses a bidirectional LSTM for better context understanding.
    *   **Output Layer**: A fully connected layer that maps LSTM outputs to the vocabulary size.
6.  **`OCRTrainer` Class**: Encapsulates the training and validation logic for the OCR model. Key functionalities include:
    *   Initialization with a model, data loaders, device, learning rate, and checkpoint directory.
    *   Definition of `CrossEntropyLoss` as the loss function and `Adam` as the optimizer.
    *   Implementation of `ReduceLROnPlateau` scheduler for adaptive learning rate adjustment.
    *   Methods for single-epoch training (`train_epoch`) and validation (`validate`).
    *   A main `train` loop that iterates over epochs, tracks losses, and implements early stopping based on validation loss.
    *   Checkpointing mechanism to save the best performing model and epoch-wise checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import logging
from pathlib import Path
from tqdm import tqdm
import time
import json

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==================== DATASET CLASS ====================
class LineOCRDataset(Dataset):
    """Load OCR dataset from split folders"""

    def __init__(self, dataset_dir, split='train', max_text_len=150, image_extensions=('.jpg', '.jpeg', '.png'), target_image_height=32, target_image_width=256, vocab_size=256):
        self.dataset_dir = Path(dataset_dir)
        self.split_dir = self.dataset_dir / split
        self.max_text_len = max_text_len
        self.image_extensions = image_extensions
        self.target_image_height = target_image_height
        self.target_image_width = target_image_width
        self.vocab_size = vocab_size # Add vocab_size to the dataset
        self.samples = []

        logger.info(f"\n📂 Loading {split.upper()} dataset...")
        logger.info(f"   Directory: {self.split_dir}")

        if not self.split_dir.exists():
            raise FileNotFoundError(f"Split directory not found: {self.split_dir}")

        # Load labels
        labels_file = self.split_dir / 'labels.txt'
        if not labels_file.exists():
            raise FileNotFoundError(f"labels.txt not found: {labels_file}")

        logger.info(f"   Reading labels from: labels.txt")

        # Parse labels
        with open(labels_file, 'r', encoding='utf-16') as f: # Changed encoding from 'utf-8' to 'utf-16'
            lines = f.readlines()

        for line in lines:
            line = line.strip()
            if not line or line.startswith('#'):
                continue

            parts = line.split(maxsplit=1)
            if len(parts) < 2:
                logger.warning(f"   ⚠️ Skipping invalid line: {line}")
                continue

            image_name = parts[0]
            text_label = parts[1].strip()
            image_path = self.split_dir / image_name

            if not image_path.exists():
                logger.warning(f"   ⚠️ Image not found: {image_path}")
                continue

            if not any(str(image_path).lower().endswith(ext) for ext in image_extensions):
                logger.warning(f"   ⚠️ Not an image file: {image_path}")
                continue

            if not text_label:
                logger.warning(f"   ⚠️ Empty label for: {image_name}")
                continue

            self.samples.append({
                'image_path': image_path,
                'text': text_label,
                'image_name': image_name
            })

        logger.info(f"   ✓ Loaded {len(self.samples)} samples with valid labels")
        if len(self.samples) == 0:
            raise ValueError(f"No valid samples found in {self.split_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            # Load image
            image = cv2.imread(str(sample['image_path']))
            if image is None:
                raise IOError(f"Cannot read image: {sample['image_path']}")

            # Resize image to a fixed size
            image = cv2.resize(image, (self.target_image_width, self.target_image_height))

            # Convert BGR to RGB
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Normalize
            image = image.astype('float32') / 255.0

            # Convert to tensor
            image = torch.from_numpy(image).permute(2, 0, 1)

            # Encode text
            text = sample['text']
            # Filter out characters whose ordinal value is >= vocab_size
            text_encoded = [ord(c) for c in text if ord(c) < self.vocab_size]

            # Pad/truncate
            if len(text_encoded) < self.max_text_len:
                text_encoded = text_encoded + [0] * (self.max_text_len - len(text_encoded))
            else:
                text_encoded = text_encoded[:self.max_text_len]

            text_tensor = torch.tensor(text_encoded, dtype=torch.long)

            return {
                'image': image,
                'text': text_tensor,
                'text_str': text,
                'image_name': sample['image_name']
            }

        except Exception as e:
            logger.error(f"Error processing {sample['image_path']}: {str(e)}")
            raise


# ==================== MODEL CLASS ====================
class CNNRNNOCRModel(nn.Module):
    """CNN + LSTM OCR Model"""

    def __init__(self, vocab_size=256, max_text_len=150):
        super().__init__()

        # CNN Backbone
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d((1, 2), (1, 2)),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.5
        )

        # Output layer
        self.fc = nn.Linear(512 * 2, vocab_size)
        self.max_text_len = max_text_len

    def forward(self, x):
        cnn_out = self.cnn(x)
        b, c, h, w = cnn_out.size()
        lstm_in = cnn_out.view(b, c, -1).permute(0, 2, 1)
        lstm_out, _ = self.lstm(lstm_in)
        output = self.fc(lstm_out)

        # Truncate output sequence length to match max_text_len for loss calculation
        if output.size(1) > self.max_text_len:
            output = output[:, :self.max_text_len, :]

        return output


# ==================== TRAINER CLASS ====================
class OCRTrainer:
    """Training orchestrator"""

    def __init__(self, model, train_loader, val_loader, device='cuda', lr=0.001, checkpoint_dir='checkpoints'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(model.parameters(), lr=lr)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=3
        ) # Removed verbose=True as it's deprecated

        self.train_losses = []
        self.val_losses = []
        self.best_val_loss = float('inf')

        logger.info(f"✅ Trainer initialized")
        logger.info(f"   Device: {device}")
        logger.info(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    def train_epoch(self, epoch):
        """Train single epoch"""
        self.model.train()
        total_loss = 0.0
        num_batches = 0

        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1} [TRAIN]", leave=True)

        for batch in pbar:
            images = batch['image'].to(self.device)
            texts = batch['text'].to(self.device)

            outputs = self.model(images)
            outputs_flat = outputs.contiguous().view(-1, outputs.size(-1))
            texts_flat = texts.view(-1)

            loss = self.criterion(outputs_flat, texts_flat)

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / num_batches
        self.train_losses.append(avg_loss)
        return avg_loss

    def validate(self):
        """Validate model"""
        self.model.eval()
        total_loss = 0.0
        num_batches = 0

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validating", leave=True):
                images = batch['image'].to(self.device)
                texts = batch['text'].to(self.device)

                outputs = self.model(images)
                outputs_flat = outputs.contiguous().view(-1, outputs.size(-1))
                texts_flat = texts.view(-1)

                loss = self.criterion(outputs_flat, texts_flat)
                total_loss += loss.item()
                num_batches += 1

        avg_loss = total_loss / num_batches
        self.val_losses.append(avg_loss)
        return avg_loss

    def train(self, epochs):
        """Train for multiple epochs"""
        logger.info(f"\n{'='*80}")
        logger.info(f"🚀 STARTING TRAINING - {epochs} EPOCHS")
        logger.info(f"{'='*80}\n")

        best_val_loss = float('inf')
        patience_count = 0
        max_patience = 5

        for epoch in range(epochs):
            start_time = time.time()

            train_loss = self.train_epoch(epoch)
            val_loss = self.validate()

            epoch_time = time.time() - start_time

            logger.info(f"\n{'─'*80}")
            logger.info(f"Epoch {epoch+1}/{epochs} | Time: {epoch_time:.1f}s")
            logger.info(f"  📊 Train Loss: {train_loss:.4f}")
            logger.info(f"  📊 Val Loss:   {val_loss:.4f}")
            logger.info(f"{'─'*80}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self.save_checkpoint(epoch, val_loss, is_best=True)
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= max_patience:
                    logger.info(f"\n⏹️  Early stopping at epoch {epoch+1}")
                    break

            self.scheduler.step(val_loss)

        logger.info(f"\n{'='*80}")
        logger.info(f"✅ TRAINING COMPLETE!")
        logger.info(f"   Best Val Loss: {best_val_loss:.4f}")
        logger.info(f"   Checkpoints: {self.checkpoint_dir}")
        logger.info(f"{'='*80}\n")

    def save_checkpoint(self, epoch, val_loss, is_best=False):
        """Save checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'val_loss': val_loss,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses
        }

        if is_best:
            path = self.checkpoint_dir / 'best_model.pt'
            logger.info(f"✓ Best model saved: {path.name}")
        else:
            path = self.checkpoint_dir / f'checkpoint_epoch_{epoch+1}.pt'

        torch.save(checkpoint, path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Main Training Execution Block

This cell orchestrates the entire OCR model training pipeline. It performs the following steps:

1.  **CUDA Error Reporting**: Sets `CUDA_LAUNCH_BLOCKING` to `1` to enable synchronous CUDA error reporting, which is beneficial for debugging GPU-related issues.
2.  **Configuration**: Defines crucial hyperparameters and paths:
    *   `DATASET_ROOT`: Specifies the root directory of the dataset in Google Drive.
    *   `CHECKPOINT_DIR`: Defines the directory where model checkpoints will be saved.
    *   `BATCH_SIZE`, `NUM_EPOCHS`, `LEARNING_RATE`: Standard training parameters.
    *   `DEVICE`: Automatically selects `cuda` if a GPU is available, otherwise falls back to `cpu`.
    *   `MAX_TEXT_LEN`: Maximum length of text labels.
    *   `TARGET_IMAGE_HEIGHT`, `TARGET_IMAGE_WIDTH`: Desired dimensions for input images.
    *   `VOCAB_SIZE`: The size of the character vocabulary (e.g., 256 for ASCII).
3.  **Dataset Loading**: Initializes `LineOCRDataset` instances for training, validation, and testing splits, passing the configured parameters. Includes robust error handling for cases where dataset directories or `labels.txt` are not found.
4.  **DataLoader Creation**: Creates `DataLoader` objects for each dataset split (`train_loader`, `val_loader`, `test_loader`). These handle batching, shuffling (for training), and multiprocessing for efficient data loading.
5.  **Model Instantiation**: Initializes the `CNNRNNOCRModel` with the specified `vocab_size` and `max_text_len`. It also logs the total and trainable parameters of the model.
6.  **Trainer Initialization**: Creates an `OCRTrainer` instance, passing the model, data loaders, device, learning rate, and checkpoint directory.
7.  **Model Training**: Invokes the `train` method of the `OCRTrainer`, commencing the training process for the specified number of epochs. This process includes epoch-wise training, validation, logging, and checkpointing.
8.  **Final Model Saving**: After training completes, the final state dictionary of the trained model is saved to `final_model.pt` in the `CHECKPOINT_DIR`.
9.  **Training Summary**: Prints a comprehensive summary of the training process, including dataset statistics, epochs trained, best validation loss achieved, and checkpoint location.

In [ ]:
import os

# Enable synchronous CUDA error reporting for debugging
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# ==================== MAIN TRAINING ====================

# 📍 IMPORTANT: Set this to your Drive path
DATASET_ROOT = '/content/drive/MyDrive/split'

# Configuration
CHECKPOINT_DIR = '/content/drive/MyDrive/Untitled Folder'
BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 0.001
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_TEXT_LEN = 150
TARGET_IMAGE_HEIGHT = 32 # New configuration parameter
TARGET_IMAGE_WIDTH = 256 # New configuration parameter
VOCAB_SIZE = 256 # Explicitly define vocab_size

logger.info(f"\n{'='*80}")
logger.info(f"OCR TRAINING PIPELINE - 70/15/15 Dataset Split (COLAB)")
logger.info(f"{'='*80}\n")

logger.info(f"⚙️  Configuration:")
logger.info(f"   Dataset: {DATASET_ROOT}")
logger.info(f"   Device: {DEVICE}")
logger.info(f"   Batch Size: {BATCH_SIZE}")
logger.info(f"   Epochs: {NUM_EPOCHS}")
logger.info(f"   Learning Rate: {LEARNING_RATE}")
logger.info(f"   Target Image Size: {TARGET_IMAGE_HEIGHT}x{TARGET_IMAGE_WIDTH}")
logger.info(f"   Vocabulary Size: {VOCAB_SIZE}\n")

# Load datasets
try:
    train_dataset = LineOCRDataset(
        dataset_dir=DATASET_ROOT,
        split='train',
        max_text_len=MAX_TEXT_LEN,
        target_image_height=TARGET_IMAGE_HEIGHT,
        target_image_width=TARGET_IMAGE_WIDTH,
        vocab_size=VOCAB_SIZE # Pass vocab_size to dataset
    )

    val_dataset = LineOCRDataset(
        dataset_dir=DATASET_ROOT,
        split='val',
        max_text_len=MAX_TEXT_LEN,
        target_image_height=TARGET_IMAGE_HEIGHT,
        target_image_width=TARGET_IMAGE_WIDTH,
        vocab_size=VOCAB_SIZE # Pass vocab_size to dataset
    )

    test_dataset = LineOCRDataset(
        dataset_dir=DATASET_ROOT,
        split='test',
        max_text_len=MAX_TEXT_LEN,
        target_image_height=TARGET_IMAGE_HEIGHT,
        target_image_width=TARGET_IMAGE_WIDTH,
        vocab_size=VOCAB_SIZE # Pass vocab_size to dataset
    )

except Exception as e:
    logger.error(f"❌ Error loading datasets: {str(e)}")
    logger.error("Please verify your dataset structure:")
    logger.error(f"  {DATASET_ROOT}/")
    logger.error(f"    ├── train/")
    logger.error(f"    │   ├── image1.jpg")
    logger.error(f"    │   └── labels.txt")
    logger.error(f"    ├── val/")
    logger.error(f"    │   └── labels.txt")
    logger.error(f"    └── test/")
    logger.error(f"        └── labels.txt")
    raise

# Create DataLoaders
logger.info(f"🔄 Creating DataLoaders...\n")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=False # Changed to False for debugging CUDA errors
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=False # Changed to False for debugging CUDA errors
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=False # Changed to False for debugging CUDA errors
)

logger.info(f"✓ Train batches: {len(train_loader)} ({len(train_dataset)} samples)")
logger.info(f"✓ Val batches: {len(val_loader)} ({len(val_dataset)} samples)")
logger.info(f"✓ Test batches: {len(test_loader)} ({len(test_dataset)} samples)\n")

# Create model
logger.info(f"🤖 Creating OCR Model (CNN + LSTM)...\n")

model = CNNRNNOCRModel(vocab_size=VOCAB_SIZE, max_text_len=MAX_TEXT_LEN)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

logger.info(f"✓ Model created")
logger.info(f"   Total parameters: {total_params:,}")
logger.info(f"   Trainable parameters: {trainable_params:,}\n")

# Train
trainer = OCRTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    lr=LEARNING_RATE,
    checkpoint_dir=CHECKPOINT_DIR
)

trainer.train(epochs=NUM_EPOCHS)

# Save final model
final_model_path = Path(CHECKPOINT_DIR) / 'final_model.pt'
torch.save(model.state_dict(), final_model_path)

logger.info(f"✅ Final model saved: {final_model_path}\n")

# Summary
logger.info(f"{'='*80}")
logger.info(f"TRAINING SUMMARY")
logger.info(f"{'='*80}")
logger.info(f"Train samples: {len(train_dataset)}")
logger.info(f"Val samples: {len(val_dataset)}")
logger.info(f"Test samples: {len(test_dataset)}")
logger.info(f"Total samples: {len(train_dataset) + len(val_dataset) + len(test_dataset)}")
logger.info(f"Epochs trained: {len(trainer.train_losses)}")
logger.info(f"Best val loss: {min(trainer.val_losses):.4f}")
logger.info(f"Checkpoints saved to: {CHECKPOINT_DIR}")
logger.info(f"{'='*80}\n")

Epoch 22 [TRAIN]:  13%|█▎        | 9/70 [00:02<00:13,  4.48it/s, loss=0.5035]